# Agroecosystem Water–Carbon–Nitrogen Analysis Workflow
**For the proposed PhD project with Prof. Fadong Li (IGSNRR, CAS)**

This reusable notebook provides starter workflows for (1) eddy-covariance data screening and daily aggregation, (2) wheat irrigation × nitrogen experiment summaries, (3) APSIM evaluation, and (4) optional machine-learning benchmarks.

No observations are included and no results are invented. Update paths and column mappings to match approved datasets. This is not a replacement for site-specific raw-flux processing, footprint analysis, or validated carbon partitioning.

## 1. Imports and configuration

In [ ]:
# If needed, uncomment:
# %pip install pandas numpy matplotlib scikit-learn openpyxl

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)

# EDIT these paths
EC_FILE = Path(r"D:\your_project\ec_half_hourly.csv")
EXPERIMENT_FILE = Path(r"D:\your_project\wheat_experiment.csv")
APSIM_FILE = Path(r"D:\your_project\apsim_vs_observed.csv")
OUTPUT_DIR = Path(r"D:\your_project\proposal_analysis_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# EC headers: adjust to the actual input file
EC = {
    "timestamp": "TIMESTAMP", "LE": "LE", "H": "H",
    "NETRAD": "NETRAD", "G": "G", "TA": "TA",
    "NEE": "NEE", "ET": None
}

# Experiment headers: adjust to the actual input file
EXP = {
    "season": "Season", "block": "Block", "irrigation": "Irrigation",
    "nitrogen": "Nitrogen_kg_ha", "cultivar": "Cultivar",
    "yield": "GrainYield_kg_ha", "biomass": "Biomass_kg_ha",
    "crop_N": "CropN_kg_ha", "ET": "ET_mm", "irrigation_mm": "Irrigation_mm"
}

# APSIM comparison headers: one row per matched observation/simulation pair
SIM = {"observed": "Observed", "simulated": "Simulated",
       "variable": "Variable", "season": "Season", "site": "Site"}


## 2. Helper functions

In [ ]:
def require_columns(df, columns, label="data"):
    missing = [c for c in columns if c and c not in df.columns]
    if missing:
        raise ValueError(f"{label}: missing required columns: {missing}")

def savefig(name):
    path = OUTPUT_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

def regression_metrics(observed, simulated):
    pair = pd.DataFrame({"obs": observed, "sim": simulated}).dropna()
    if len(pair) < 2:
        return {"n": len(pair), "Bias": np.nan, "MAE": np.nan, "RMSE": np.nan, "R2": np.nan}
    obs, sim = pair["obs"].to_numpy(), pair["sim"].to_numpy()
    ss_tot = np.sum((obs - obs.mean()) ** 2)
    return {
        "n": len(pair),
        "Bias": float(np.mean(sim - obs)),
        "MAE": float(np.mean(np.abs(sim - obs))),
        "RMSE": float(np.sqrt(np.mean((sim - obs) ** 2))),
        "R2": float(1 - np.sum((obs - sim) ** 2) / ss_tot) if ss_tot > 0 else np.nan
    }

def le_to_et_mm_per_interval(LE_W_m2, interval_minutes=30, temperature_C=None):
    # ET (mm per interval) = LE * seconds / latent heat of vaporization.
    # If temperature is missing, lambda at 20 C is used as an explicit approximation.
    lam = 2.501e6 if temperature_C is None else (2.501 - 0.002361 * np.asarray(temperature_C)) * 1e6
    return np.asarray(LE_W_m2) * (interval_minutes * 60) / lam


## 3. Eddy covariance screening and daily summaries
Assumes processed half-hourly fluxes. Confirm units, timestamp format/time zone, sign conventions, and the definition of G. If G already includes soil heat storage, do not subtract storage a second time.

In [ ]:
if EC_FILE.exists():
    ec = pd.read_csv(EC_FILE)
    require_columns(ec, [EC["timestamp"], EC["LE"], EC["H"], EC["NETRAD"], EC["G"]], "EC")
    ec[EC["timestamp"]] = pd.to_datetime(ec[EC["timestamp"]], errors="coerce", dayfirst=True)
    ec = ec.dropna(subset=[EC["timestamp"]]).sort_values(EC["timestamp"]).copy()

    print("Rows:", len(ec))
    print("Time span:", ec[EC["timestamp"]].min(), "to", ec[EC["timestamp"]].max())
    print("Duplicate timestamps:", int(ec[EC["timestamp"]].duplicated().sum()))
    print("Most common timestamp intervals:")
    display(ec[EC["timestamp"]].diff().value_counts().head(10))

    temperature = ec[EC["TA"]] if EC["TA"] in ec.columns else None
    ec["ET_mm_interval"] = le_to_et_mm_per_interval(ec[EC["LE"]], 30, temperature)

    # Available energy and residual; G is surface soil heat flux.
    ec["AvailableEnergy_Wm2"] = ec[EC["NETRAD"]] - ec[EC["G"]]
    ec["EBR_residual_Wm2"] = ec["AvailableEnergy_Wm2"] - ec[EC["LE"]] - ec[EC["H"]]
    ec["basic_valid"] = ec[[EC["LE"], EC["H"], EC["NETRAD"], EC["G"]]].notna().all(axis=1)
    ec.loc[~ec["basic_valid"], ["ET_mm_interval", "AvailableEnergy_Wm2", "EBR_residual_Wm2"]] = np.nan
    ec["date"] = ec[EC["timestamp"]].dt.floor("D")

    daily = ec.groupby("date").agg(
        records=("basic_valid", "size"),
        valid_records=("basic_valid", "sum"),
        ET_mm=("ET_mm_interval", "sum"),
        mean_EBR_residual_Wm2=("EBR_residual_Wm2", "mean"),
        mean_LE_Wm2=(EC["LE"], "mean"),
        mean_H_Wm2=(EC["H"], "mean")
    )
    daily["coverage_fraction"] = daily["valid_records"] / daily["records"]
    # Transparent starter threshold: at least 80% of 48 expected half-hourly intervals.
    daily.loc[daily["valid_records"] < 0.80 * 48, "ET_mm"] = np.nan

    ec.to_csv(OUTPUT_DIR / "ec_screened_intervals.csv", index=False)
    daily.to_csv(OUTPUT_DIR / "ec_daily_summary.csv")
    display(daily.head())
else:
    print("EC file not found. Edit EC_FILE to run this section.")


In [ ]:
if EC_FILE.exists():
    plt.figure(figsize=(10, 4))
    plt.plot(daily.index, daily["ET_mm"], linewidth=0.8)
    plt.xlabel("Date")
    plt.ylabel("ET (mm day-1; screened daily totals)")
    plt.title("Daily evapotranspiration from LE")
    savefig("ec_daily_et.png")

    valid = ec[["AvailableEnergy_Wm2", EC["LE"], EC["H"]]].dropna()
    if len(valid):
        turbulent = valid[EC["LE"]] + valid[EC["H"]]
        plt.figure(figsize=(6, 5))
        plt.scatter(valid["AvailableEnergy_Wm2"], turbulent, s=8, alpha=0.4)
        lo = min(valid["AvailableEnergy_Wm2"].min(), turbulent.min())
        hi = max(valid["AvailableEnergy_Wm2"].max(), turbulent.max())
        plt.plot([lo, hi], [lo, hi], linestyle="--")
        plt.xlabel("Available energy (NETRAD - G), W m-2")
        plt.ylabel("LE + H, W m-2")
        plt.title("Energy-balance comparison")
        savefig("ec_energy_balance.png")


## 4. Wheat experiment treatment summaries
This is descriptive analysis only. Final inference must respect the split-plot randomization and its error strata.

In [ ]:
if EXPERIMENT_FILE.exists():
    exp = pd.read_csv(EXPERIMENT_FILE)
    required = [EXP["season"], EXP["block"], EXP["irrigation"], EXP["nitrogen"], EXP["yield"]]
    require_columns(exp, required, "Experiment")
    group_cols = [EXP["season"], EXP["irrigation"], EXP["nitrogen"]]
    summary = exp.groupby(group_cols, dropna=False)[EXP["yield"]].agg(["count", "mean", "std"]).reset_index()
    summary.to_csv(OUTPUT_DIR / "experiment_treatment_summary.csv", index=False)
    display(summary)

    plt.figure(figsize=(9, 5))
    for irrigation, sub in exp.groupby(EXP["irrigation"]):
        means = sub.groupby(EXP["nitrogen"])[EXP["yield"]].mean()
        plt.plot(means.index.astype(str), means.values, marker="o", label=str(irrigation))
    plt.xlabel("Nitrogen rate / treatment")
    plt.ylabel("Grain yield (input units)")
    plt.title("Mean yield by irrigation and nitrogen")
    plt.legend(title="Irrigation")
    savefig("experiment_yield_treatment_means.png")

    if EXP["ET"] in exp.columns:
        # If yield is kg/ha and ET is mm: 1 mm over 1 ha = 10 m3.
        exp["WaterProductivity_kg_m3"] = np.where(exp[EXP["ET"]] > 0, exp[EXP["yield"]] / (10 * exp[EXP["ET"]]), np.nan)
        exp.to_csv(OUTPUT_DIR / "experiment_with_water_productivity.csv", index=False)
else:
    print("Experiment file not found. Edit EXPERIMENT_FILE to run this section.")


## 5. APSIM evaluation
Use paired observations and simulations with identical units and matching dates/plots. Do not evaluate on calibration data alone.

In [ ]:
if APSIM_FILE.exists():
    simdata = pd.read_csv(APSIM_FILE)
    require_columns(simdata, [SIM["observed"], SIM["simulated"], SIM["variable"]], "APSIM")
    metrics = []
    for variable, sub in simdata.groupby(SIM["variable"]):
        metrics.append({"Variable": variable, **regression_metrics(sub[SIM["observed"]], sub[SIM["simulated"]])})
    metrics_df = pd.DataFrame(metrics)
    metrics_df.to_csv(OUTPUT_DIR / "apsim_evaluation_metrics.csv", index=False)
    display(metrics_df)

    for variable, sub in simdata.groupby(SIM["variable"]):
        pair = sub[[SIM["observed"], SIM["simulated"]]].dropna()
        if pair.empty:
            continue
        plt.figure(figsize=(5.5, 5))
        plt.scatter(pair[SIM["observed"]], pair[SIM["simulated"]], alpha=0.65)
        lo = min(pair[SIM["observed"]].min(), pair[SIM["simulated"]].min())
        hi = max(pair[SIM["observed"]].max(), pair[SIM["simulated"]].max())
        plt.plot([lo, hi], [lo, hi], linestyle="--")
        plt.xlabel(f"Observed {variable}")
        plt.ylabel(f"Simulated {variable}")
        plt.title(f"APSIM evaluation: {variable}")
        savefig(f"apsim_evaluation_{str(variable).replace(' ', '_')}.png")
else:
    print("APSIM comparison file not found. Edit APSIM_FILE to run this section.")


## 6. Optional machine-learning benchmark
The example below is exploratory and uses a random split. Final analysis should use grouped/leave-one-season-out validation to prevent information leakage between plots, seasons, and repeated observations.

In [ ]:
if EXPERIMENT_FILE.exists():
    from sklearn.model_selection import train_test_split
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer
    from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
    from sklearn.svm import SVR

    exp = pd.read_csv(EXPERIMENT_FILE)
    target = EXP["yield"]
    candidate_features = [EXP["irrigation"], EXP["nitrogen"], EXP["cultivar"], EXP["season"]]
    features = [c for c in candidate_features if c in exp.columns]
    ml = exp[features + [target]].dropna(subset=[target]).copy()

    if len(ml) >= 12 and features:
        X, y = ml[features], ml[target]
        categorical = [c for c in features if X[c].dtype == "object"]
        numeric = [c for c in features if c not in categorical]
        preprocess = ColumnTransformer([
            ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                              ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical),
            ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                              ("scale", StandardScaler())]), numeric)
        ])
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
        models = {
            "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42),
            "GradientBoosting": GradientBoostingRegressor(random_state=42),
            "SVR": SVR(C=10, epsilon=0.1)
        }
        rows = []
        for name, model in models.items():
            pipe = Pipeline([("preprocess", preprocess), ("model", model)])
            pipe.fit(X_train, y_train)
            pred = pipe.predict(X_test)
            rows.append({"Model": name, **regression_metrics(y_test, pred)})
        ml_metrics = pd.DataFrame(rows)
        ml_metrics.to_csv(OUTPUT_DIR / "ml_exploratory_benchmark.csv", index=False)
        display(ml_metrics)
        print("Reminder: replace random splitting with season/group-blocked validation for publishable results.")
    else:
        print("Insufficient rows or features for the exploratory ML benchmark.")


## 7. Export available CSV outputs to an Excel workbook

In [ ]:
xlsx_path = OUTPUT_DIR / "proposal_analysis_outputs.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    for csv_path in sorted(OUTPUT_DIR.glob("*.csv")):
        try:
            frame = pd.read_csv(csv_path)
            frame.to_excel(writer, sheet_name=csv_path.stem[:31], index=False)
        except Exception as exc:
            print("Skipped", csv_path.name, "-", exc)

print("Output folder:", OUTPUT_DIR)
print("Workbook:", xlsx_path)
print("Before publication, document file versions, units, time zone, QC criteria, gap-filling method, footprint representativeness, model version, calibration data and independent validation design.")
